# LASTDANCE — ASR dev-subset-5 gate

Notebook này chạy tuần tự Whisper Large-v3 và PhoWhisper-large trên **Tesla T4**. Mỗi model phải intentional-stop sau hai video rồi được một process mới resume. Gate ghi peak CUDA, runtime/model/weight provenance, transcript có timestamp và upload evidence vào private Hugging Face Dataset. Gate PASS vẫn giữ `production_model_selected=false` cho tới khi nghe review và người dùng chốt model.


## Chuẩn bị trên Kaggle

- Bật **Internet** và chọn accelerator **Tesla T4**; dùng tài khoản GPU Nhánh 3, không tranh quota Visual.
- Gắn private Dataset `lastdance-asr-dev-audio` có đúng `dev-gate-audio-report.json`, `manifests/` và năm WAV canonical. Không cần gắn Dataset source code: notebook clone nhánh ASR từ GitHub giống notebook SigLIP.
- Bật Kaggle Secret `HF_TOKEN` có quyền tạo/ghi private Dataset trong tài khoản Hugging Face.
- Dùng session sạch. Không gắn transcript/evidence cũ vào `/kaggle/input`; resume gate được chứng minh trong `/kaggle/working` bằng process mới.


## 1. Clone và khóa đúng commit ASR


In [ ]:
from pathlib import Path
import os
import socket
import subprocess
import sys

WORKING_ROOT = Path('/kaggle/working')
REPO = WORKING_ROOT / 'LASTDANCE'
BRANCH = 'codex/offline-asr'
EXPECTED_COMMIT = 'a6a81bcf2dfb8785668eb307923c5ae75a3f83ab'
CLONE_URL = 'https://' + 'github.com/ThanhVu165/LASTDANCE.git'

if len(EXPECTED_COMMIT) != 40 or any(char not in '0123456789abcdef' for char in EXPECTED_COMMIT):
    raise RuntimeError('Notebook ASR chưa pin commit runner. Commit/push code ASR rồi cập nhật EXPECTED_COMMIT trước khi chạy Kaggle.')
assert not any(marker in CLONE_URL for marker in '[]()'), CLONE_URL

try:
    socket.getaddrinfo('github.com', 443)
except socket.gaierror as error:
    raise RuntimeError('Không phân giải được github.com. Bật Internet trong Kaggle Settings rồi khởi động session mới.') from error

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} tồn tại nhưng không phải Git repo; hãy dùng session Kaggle sạch.')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', CLONE_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPO, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
for required in (
    'scripts/run_asr_dev_gate.py',
    'scripts/compare_asr_dev_gate.py',
    'requirements/asr-kaggle-gpu.txt',
    'configs/asr_models.json',
):
    assert (REPO / required).is_file(), required
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('CODE CHECKOUT PASS:', actual_commit)


## 2. Cài dependency Kaggle GPU


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/kaggle/working/LASTDANCE')
assert (REPO / 'requirements/asr-kaggle-gpu.txt').is_file(), REPO
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', 'requirements/asr-kaggle-gpu.txt'], cwd=REPO, check=True)
print('DEPENDENCY INSTALL PASS')


## 3. Xác thực HF, T4 và model registry

Token chỉ được đọc từ Kaggle Secret và export cho các subprocess tải model; notebook không in token. CUDA phải là T4, Torch phải đủ an toàn cho PhoWhisper `.bin`, và hai model vẫn chỉ được phép Dev Gate.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO = Path('/kaggle/working/LASTDANCE')
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.strip(), 'Thiếu Kaggle Secret HF_TOKEN'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/kaggle/working/huggingface-cache'
subprocess.run([sys.executable, '-m', 'scripts.environment_doctor', '--profile', 'asr-kaggle-gpu', '--skip-data'], cwd=REPO, check=True, env=os.environ.copy())

import torch
from offline.asr_models import load_asr_model_config

assert torch.cuda.is_available(), 'CUDA unavailable; CPU fallback is forbidden'
gpu_name = torch.cuda.get_device_name()
assert 'T4' in gpu_name, gpu_name
assert tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 6), torch.__version__
for model_key in ('whisper_large_v3', 'phowhisper_large'):
    row = load_asr_model_config(model_key)
    assert row['dev_gate_allowed'] is True
    assert row['production_allowed'] is False
print('ASR RUNTIME PREFLIGHT PASS:', gpu_name, torch.__version__)


## 4. Resolve và verify đúng Dataset audio 5 video

Notebook chỉ nhận đúng một report dưới `/kaggle/input`, kiểm codec PCM s16le, 16 kHz mono, SHA-bound manifest và đúng dev subset cố định.


In [ ]:
from pathlib import Path
import json

INPUT_ROOT = Path('/kaggle/input')
EXPECTED_VIDEO_IDS = ['L21_V001', 'L21_V002', 'L21_V003', 'L21_V005', 'L21_V006']
MODELS = ['whisper_large_v3', 'phowhisper_large']
audio_reports = list(INPUT_ROOT.rglob('dev-gate-audio-report.json'))
assert len(audio_reports) == 1, f'expected one audio input, found {len(audio_reports)}: {audio_reports}'
AUDIO_ROOT = audio_reports[0].parent
audio_report = json.loads(audio_reports[0].read_text(encoding='utf-8'))
assert audio_report['video_ids'] == EXPECTED_VIDEO_IDS
assert audio_report['ready_count'] == 5 and audio_report['no_audio_count'] == 0
assert 1.8 <= float(audio_report['mean_megabytes_per_minute']) <= 2.1
for video_id in EXPECTED_VIDEO_IDS:
    manifest = json.loads((AUDIO_ROOT / 'manifests' / f'{video_id}.json').read_text(encoding='utf-8'))
    assert manifest['status'] == 'ready'
    assert manifest['codec'] == 'pcm_s16le'
    assert manifest['sample_rate_hz'] == 16000 and manifest['channels'] == 1
    wav_path = AUDIO_ROOT / manifest['wav_path']
    assert wav_path.is_file() and wav_path.stat().st_size > 0, wav_path
print('AUDIO PREFLIGHT PASS:', AUDIO_ROOT, audio_report['audio_duration_seconds'], 'seconds')


## 5. Chạy hai model với intentional interruption và process mới resume

Mỗi invocation là subprocess riêng. Nếu checkpoint partial đã tồn tại trong session, notebook không tạo interruption thứ hai mà chạy process resume. Manifest hoàn chỉnh hợp lệ được reuse khi chạy lại cell.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/kaggle/working/LASTDANCE')
OUTPUT_ROOT = Path('/kaggle/working/asr-transcripts')

def run_model_gate(model_key):
    artifact_dir = OUTPUT_ROOT / 'dev-subset-5' / model_key
    checkpoint_path = artifact_dir / 'checkpoint.json'
    manifest_path = artifact_dir / 'manifest.json'
    base = [
        sys.executable, '-m', 'scripts.run_asr_dev_gate',
        '--model', model_key,
        '--audio-root', str(AUDIO_ROOT),
        '--output-root', str(OUTPUT_ROOT),
    ]
    if not manifest_path.is_file() and not checkpoint_path.is_file():
        interrupted = subprocess.run(base + ['--stop-after-videos', '2'], cwd=REPO, check=False, env=os.environ.copy())
        assert interrupted.returncode == 75, (model_key, interrupted.returncode)
        assert checkpoint_path.is_file(), checkpoint_path
    resumed = subprocess.run(base + ['--require-resume-verified'], cwd=REPO, check=False, env=os.environ.copy())
    assert resumed.returncode == 0, (model_key, resumed.returncode)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest['complete'] is True
    assert manifest['video_ids'] == EXPECTED_VIDEO_IDS
    assert manifest['record_count'] == 5
    assert manifest['checkpoint_resume_verified'] is True
    assert manifest['runtime']['device'] == 'cuda'
    assert 'T4' in manifest['runtime']['gpu_name']
    assert manifest['runtime']['peak_cuda_memory_bytes'] > 0
    return manifest_path

manifest_paths = {model_key: run_model_gate(model_key) for model_key in MODELS}
print('BOTH ASR MODEL GATES PASS')


## 6. So sánh và tạo gate report

WER chỉ thêm khi đã có ground truth. Report mặc định giữ manual listening bắt buộc và không tự chọn production model.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

COMPARISON = Path('/kaggle/working/asr-dev-gate-comparison.json')
GATE_REPORT = Path('/kaggle/working/asr-dev-gate-report.json')
command = [
    sys.executable, '-m', 'scripts.compare_asr_dev_gate',
    '--whisper-manifest', str(manifest_paths['whisper_large_v3']),
    '--phowhisper-manifest', str(manifest_paths['phowhisper_large']),
    '--output', str(COMPARISON),
]
# Nếu đã có ground truth sẵn: command += ['--ground-truth', '/kaggle/input/.../ground-truth.json']
subprocess.run(command, cwd=REPO, check=True, env=os.environ.copy())
comparison = json.loads(COMPARISON.read_text(encoding='utf-8'))
assert comparison['video_ids'] == EXPECTED_VIDEO_IDS
assert comparison['manual_review_required'] is True
assert comparison['production_model_selected'] is False
manifests = {key: json.loads(path.read_text(encoding='utf-8')) for key, path in manifest_paths.items()}
gate_report = {
    'schema_version': 1,
    'complete': True,
    'code_commit': actual_commit,
    'video_ids': EXPECTED_VIDEO_IDS,
    'models': {key: {'model': value['model'], 'runtime': value['runtime'], 'record_count': value['record_count'], 'checkpoint_resume_verified': value['checkpoint_resume_verified']} for key, value in manifests.items()},
    'manual_review_required': True,
    'production_model_selected': False,
}
GATE_REPORT.write_text(json.dumps(gate_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
for model_key in MODELS:
    row = comparison['models'][model_key]
    print(model_key, 'RTF=', row['real_time_factor'], 'peak=', row['peak_cuda_memory_bytes'])
print('ASR GATE REPORT PASS:', GATE_REPORT)
print('MANUAL LISTENING REVIEW IS STILL REQUIRED')


## 7. Archive và upload evidence lên private HF Dataset

Cell này chỉ chạy sau khi hai model gate và report đã PASS. Nó atomic-upload archive + checksum vào `<HF_USER>/lastdance-asr-artifacts/asr/dev-gate/dev-subset-5/`, tải checksum lại để verify và không chứa WAV, video, model weights/cache hay token. Đây là evidence Dev Gate, không phải production artifact.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import tarfile

from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download

HF_REPO_NAME = 'lastdance-asr-artifacts'
HF_REPO_TYPE = 'dataset'
REMOTE_ROOT = 'asr/dev-gate/dev-subset-5'
ARCHIVE = Path('/kaggle/working') / f'lastdance-asr-dev-gate-{actual_commit[:7]}.tar.gz'
CHECKSUM = ARCHIVE.with_suffix(ARCHIVE.suffix + '.sha256')

gate_report = json.loads(GATE_REPORT.read_text(encoding='utf-8'))
assert gate_report['complete'] is True
assert gate_report['code_commit'] == actual_commit
assert gate_report['manual_review_required'] is True
assert gate_report['production_model_selected'] is False
if not ARCHIVE.exists():
    with tarfile.open(ARCHIVE, 'w:gz') as archive:
        archive.add(OUTPUT_ROOT, arcname='asr-transcripts')
        archive.add(COMPARISON, arcname=COMPARISON.name)
        archive.add(GATE_REPORT, arcname=GATE_REPORT.name)

with tarfile.open(ARCHIVE, 'r:gz') as archive:
    members = archive.getmembers()
    names = {member.name for member in members}
    for member in members:
        member_path = PurePosixPath(member.name)
        assert not member_path.is_absolute(), member.name
        assert '..' not in member_path.parts, member.name
        assert not member.issym() and not member.islnk(), member.name
    assert COMPARISON.name in names and GATE_REPORT.name in names
    for model_key in MODELS:
        assert f'asr-transcripts/dev-subset-5/{model_key}/manifest.json' in names
    forbidden = {'.wav', '.mp4', '.bin', '.safetensors'}
    assert not any(PurePosixPath(name).suffix.lower() in forbidden for name in names)

digest = hashlib.sha256()
with ARCHIVE.open('rb') as source:
    for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
archive_sha256 = digest.hexdigest()
CHECKSUM.write_text(f'{archive_sha256}  {ARCHIVE.name}\n', encoding='utf-8')

assert HF_TOKEN and HF_TOKEN.strip(), 'Thiếu Kaggle Secret HF_TOKEN'
api = HfApi(token=HF_TOKEN)
hf_user = api.whoami()['name']
HF_REPO_ID = f'{hf_user}/{HF_REPO_NAME}'
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, private=True, exist_ok=True)
repo_info = api.repo_info(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
assert repo_info.private is True, f'{HF_REPO_ID} phải là private Dataset'
remote_archive = f'{REMOTE_ROOT}/{ARCHIVE.name}'
remote_checksum = f'{REMOTE_ROOT}/{CHECKSUM.name}'
remote_files = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE))
has_archive = remote_archive in remote_files
has_checksum = remote_checksum in remote_files
assert has_archive == has_checksum, 'HF evidence dở dang: archive/checksum không đồng bộ'
if not has_archive:
    commit_info = api.create_commit(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        operations=[
            CommitOperationAdd(path_in_repo=remote_archive, path_or_fileobj=str(ARCHIVE)),
            CommitOperationAdd(path_in_repo=remote_checksum, path_or_fileobj=str(CHECKSUM)),
        ],
        commit_message=f'data(asr): preserve dev-subset-5 gate evidence {actual_commit[:7]}',
    )
    hf_commit = commit_info.oid
else:
    hf_commit = repo_info.sha
downloaded_checksum = Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, filename=remote_checksum, token=HF_TOKEN, force_download=True))
remote_sha256 = downloaded_checksum.read_text(encoding='utf-8').split()[0].lower()
assert remote_sha256 == archive_sha256, (remote_sha256, archive_sha256)
print('ASR DEV GATE EVIDENCE UPLOAD PASS')
print('HF repo:', HF_REPO_ID)
print('HF commit:', hf_commit)
print('remote archive:', remote_archive)
print('sha256:', archive_sha256)
print('production_model_selected=false; manual listening review vẫn bắt buộc')
